# Generation

This notebook builds timeline data and renders GIF outputs. For large datasets, render one GIF per day instead of one full-timeline GIF.


In [1]:
from pathlib import Path
from types import SimpleNamespace
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import Image as DisplayImage, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import config
from timeline import build_timeline
from animate_timeline import build_animation, build_daily_animations


In [2]:
timeline_args = SimpleNamespace(
    input_csv=config.INTEGRATED_OBJECTS_CSV,
    output_csv=config.OBJECT_TIMELINE_CSV,
)

timeline_rows = build_timeline(timeline_args)
len(timeline_rows), timeline_rows[:3]

(69933,
 [{'image_name': 'IR_57327.jpg',
   'timestamp': '2025-04-23 13:28:31',
   'detection_index': '4',
   'label': 'Machine',
   'people_or_machine': 'machine',
   'object_id': 'machine_2',
   'canonical_label': 'Machine',
   'merge_role': 'keep',
   'bbox_x0': '103.119843',
   'bbox_y0': '175.158752',
   'bbox_x1': '145.050507',
   'bbox_y1': '270.151733',
   'projected_x': '6.725138',
   'projected_y': '0.443425',
   'anchor_x': '6.476916',
   'anchor_y': '0.453659',
   'display_x': '6.476916',
   'display_y': '0.453659',
   'position_source': 'dbscan_machine_global_median_anchor',
   'static_cluster_id': 'dbscan_machine_0000',
   'clustering_method': 'dbscan_machine',
   'temp_mean_c': '25.472376',
   'temp_max_c': '31.236292'},
  {'image_name': 'IR_57327.jpg',
   'timestamp': '2025-04-23 13:28:31',
   'detection_index': '5',
   'label': 'Person',
   'people_or_machine': 'person',
   'object_id': 'person_frame_00003',
   'canonical_label': 'Person',
   'merge_role': 'keep',
   '

In [3]:
'''
RUN_DAILY_ANIMATION = True
DISPLAY_FIRST_GIF = False  # Rendering GIFs inside Jupyter can be heavy; keep this False for full runs.

animation_args = SimpleNamespace(
    input_csv=config.OBJECT_TIMELINE_CSV,
    layout=config.LAYOUT_IMAGE,
    output=config.TIMELINE_GIF,
    daily_output_dir=config.DAILY_ANIMATION_OUTPUT_DIR,
    map_width=15.0,
    map_height=12.0,
    fps=4,
    temperature_column='temp_mean_c',
    machine_min_observations=2,
    people_min_observations=1,
    max_frames=None,
    day=None,
    split_by_day=True,
)

if RUN_DAILY_ANIMATION:
    gif_paths = build_daily_animations(animation_args)
    print(f"Generated {len(gif_paths)} daily GIFs")
    for path in gif_paths:
        print(path)
    if DISPLAY_FIRST_GIF and gif_paths:
        display(DisplayImage(filename=str(gif_paths[0])))
else:
    print('Daily animation not rendered. Set RUN_DAILY_ANIMATION = True to generate daily GIFs.')

'''

'\nRUN_DAILY_ANIMATION = True\nDISPLAY_FIRST_GIF = False  # Rendering GIFs inside Jupyter can be heavy; keep this False for full runs.\n\nanimation_args = SimpleNamespace(\n    input_csv=config.OBJECT_TIMELINE_CSV,\n    layout=config.LAYOUT_IMAGE,\n    output=config.TIMELINE_GIF,\n    daily_output_dir=config.DAILY_ANIMATION_OUTPUT_DIR,\n    map_width=15.0,\n    map_height=12.0,\n    fps=4,\n    temperature_column=\'temp_mean_c\',\n    machine_min_observations=2,\n    people_min_observations=1,\n    max_frames=None,\n    day=None,\n    split_by_day=True,\n)\n\nif RUN_DAILY_ANIMATION:\n    gif_paths = build_daily_animations(animation_args)\n    print(f"Generated {len(gif_paths)} daily GIFs")\n    for path in gif_paths:\n        print(path)\n    if DISPLAY_FIRST_GIF and gif_paths:\n        display(DisplayImage(filename=str(gif_paths[0])))\nelse:\n    print(\'Daily animation not rendered. Set RUN_DAILY_ANIMATION = True to generate daily GIFs.\')\n\n'

## Timeline Quality Debug

Use this area after rebuilding the timeline to check whether a 07 spike already exists in the 06 timeline CSV. It checks per-object/per-day temperature jumps, duplicate timestamps, long gaps, and source rows from the 05 integrated CSV.


In [4]:
TIMELINE_TEMP_COLUMN = 'temp_mean_c'
JUMP_THRESHOLD_C = 1.0
GAP_THRESHOLD_MIN = 5.0

quality_df = pd.read_csv(config.OBJECT_TIMELINE_CSV)
quality_df['timestamp_dt'] = pd.to_datetime(quality_df['timestamp'])
quality_df[TIMELINE_TEMP_COLUMN] = pd.to_numeric(quality_df[TIMELINE_TEMP_COLUMN], errors='coerce')
quality_static = quality_df[quality_df['people_or_machine'] == 'machine'].copy()
quality_static['date'] = quality_static['timestamp_dt'].dt.date.astype(str)
quality_static = quality_static.sort_values(['object_id', 'timestamp_dt'])
quality_static['temp_delta_c'] = quality_static.groupby('object_id')[TIMELINE_TEMP_COLUMN].diff()
quality_static['abs_temp_delta_c'] = quality_static['temp_delta_c'].abs()
quality_static['gap_min'] = quality_static.groupby('object_id')['timestamp_dt'].diff().dt.total_seconds() / 60.0

def p95(series):
    series = series.dropna()
    return series.quantile(0.95) if len(series) else np.nan

object_day_quality = (
    quality_static
    .groupby(['date', 'object_id', 'canonical_label'], dropna=False)
    .agg(
        rows=(TIMELINE_TEMP_COLUMN, 'size'),
        unique_timestamps=('timestamp_dt', 'nunique'),
        temp_min_c=(TIMELINE_TEMP_COLUMN, 'min'),
        temp_max_c=(TIMELINE_TEMP_COLUMN, 'max'),
        temp_range_c=(TIMELINE_TEMP_COLUMN, lambda s: s.max() - s.min()),
        max_abs_delta_c=('abs_temp_delta_c', 'max'),
        p95_abs_delta_c=('abs_temp_delta_c', p95),
        max_gap_min=('gap_min', 'max'),
    )
    .reset_index()
)
object_day_quality['duplicate_timestamp_rows'] = object_day_quality['rows'] - object_day_quality['unique_timestamps']
object_day_quality = object_day_quality.sort_values(['max_abs_delta_c', 'temp_range_c'], ascending=False)
display(object_day_quality.head(30))


,date,object_id,canonical_label,rows,unique_timestamps,temp_min_c,temp_max_c,temp_range_c,max_abs_delta_c,p95_abs_delta_c,max_gap_min,duplicate_timestamp_rows
76,2025-05-01,screen_1,Screen,985,984,21.312052,28.809393,7.497341,4.949690,1.271038,1035.016667,1
106,2025-05-07,cableduct_4,Cableduct,875,873,21.503016,27.153057,5.650041,4.645969,1.232100,940.033333,2
50,2025-04-29,screen_1,Screen,1227,1100,21.325386,27.434917,6.109531,4.392540,1.633965,5235.050000,127
29,2025-04-25,cableduct_3,Cableduct,1549,827,19.590124,26.060154,6.470030,4.302942,0.836563,978.833333,722
51,2025-04-29,screen_2,Screen,1011,1011,20.728750,25.833685,5.104935,4.199076,0.768914,5235.050000,0
93,2025-05-06,cableduct_3,Cableduct,423,338,21.193510,26.510857,5.317347,4.172593,1.159821,5123.833333,85
41,2025-04-29,cableduct_3,Cableduct,600,495,21.391724,25.390348,3.998624,4.094329,0.964068,5328.483333,105
88,2025-05-02,screen_1,Screen,914,914,21.181215,26.790165,5.608950,4.033272,1.317981,987.550000,0
101,2025-05-06,screen_2,Screen,506,506,20.897305,26.547829,5.650524,4.002335,0.894718,5125.350000,0
105,2025-05-07,cableduct_3,Cableduct,1061,742,20.932367,26.231953,5.299586,3.902060,1.130413,939.516667,319


In [5]:
# Detailed check for the object/date that looks strange in 07.
DEBUG_OBJECT_ID = 'cableduct_2'
DEBUG_DATE = '2025-04-25'
DEBUG_TOP_N = 30

object_day = quality_static[
    (quality_static['object_id'] == DEBUG_OBJECT_ID) &
    (quality_static['date'] == DEBUG_DATE)
].sort_values('timestamp_dt').copy()

print(f"{DEBUG_DATE} | {DEBUG_OBJECT_ID} rows: {len(object_day)}")
print(f"unique timestamps: {object_day['timestamp_dt'].nunique()}")
if len(object_day):
    print(f"temp range: {object_day[TIMELINE_TEMP_COLUMN].min():.3f} to {object_day[TIMELINE_TEMP_COLUMN].max():.3f} C")
    print(f"max abs delta: {object_day['abs_temp_delta_c'].max():.3f} C")
    print(f"max gap: {object_day['gap_min'].max():.2f} min")

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(object_day['timestamp_dt'], object_day[TIMELINE_TEMP_COLUMN], color='#1f77b4', linewidth=1.4, label=TIMELINE_TEMP_COLUMN)
jump_rows = object_day[object_day['abs_temp_delta_c'] >= JUMP_THRESHOLD_C]
if not jump_rows.empty:
    ax.scatter(jump_rows['timestamp_dt'], jump_rows[TIMELINE_TEMP_COLUMN], color='red', s=32, label=f'jump >= {JUMP_THRESHOLD_C} C')
ax.set_title(f'{DEBUG_DATE} | {DEBUG_OBJECT_ID} timeline temperature check')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Temperature (C)')
ax.grid(alpha=0.25)
ax.legend(loc='best')
fig.autofmt_xdate(rotation=25)
fig.tight_layout()

inspect_cols = [
    'timestamp_dt', 'timestamp', 'image_name', 'detection_index', 'label', 'canonical_label', 'object_id',
    'temp_mean_c', 'temp_max_c', 'temp_delta_c', 'abs_temp_delta_c', 'gap_min',
    'projected_x', 'projected_y', 'anchor_x', 'anchor_y', 'display_x', 'display_y',
    'position_source', 'static_cluster_id', 'clustering_method',
]
inspect_cols = [col for col in inspect_cols if col in object_day.columns]
print('Largest jumps:')
display(object_day.sort_values('abs_temp_delta_c', ascending=False)[inspect_cols].head(DEBUG_TOP_N))
print('Hottest rows:')
display(object_day.sort_values(TIMELINE_TEMP_COLUMN, ascending=False)[inspect_cols].head(DEBUG_TOP_N))


2025-04-25 | cableduct_2 rows: 964
unique timestamps: 898
temp range: 19.719 to 24.698 C
max abs delta: 3.705 C
max gap: 989.85 min
Largest jumps:


,timestamp_dt,timestamp,image_name,detection_index,label,canonical_label,object_id,temp_mean_c,temp_max_c,temp_delta_c,...,gap_min,projected_x,projected_y,anchor_x,anchor_y,display_x,display_y,position_source,static_cluster_id,clustering_method
21284,2025-04-25 07:35:39,2025-04-25 07:35:39,IR_59349.jpg,5,Cableduct,Cableduct,cableduct_2,19.790119,22.299999,-3.704954,...,989.850000,9.105911,2.837789,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
28207,2025-04-25 13:53:09,2025-04-25 13:53:09,IR_60104.jpg,6,Cableduct,Cableduct,cableduct_2,21.286543,23.900000,-3.168611,...,0.516667,7.239231,3.327463,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
27896,2025-04-25 13:33:39,2025-04-25 13:33:39,IR_60065.jpg,10,Cableduct,Cableduct,cableduct_2,22.031336,27.400000,-2.544208,...,1.016667,7.460018,3.405265,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
28704,2025-04-25 14:22:09,2025-04-25 14:22:09,IR_60162.jpg,7,Cableduct,Cableduct,cableduct_2,21.291300,24.100000,-2.519813,...,0.500000,7.453705,3.400092,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
27799,2025-04-25 13:27:38,2025-04-25 13:27:38,IR_60053.jpg,6,Cableduct,Cableduct,cableduct_2,24.558969,33.200001,2.443660,...,0.500000,7.338344,3.350806,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
28285,2025-04-25 13:57:39,2025-04-25 13:57:39,IR_60113.jpg,6,Cableduct,Cableduct,cableduct_2,21.465883,24.000000,-2.178835,...,0.516667,7.307934,3.339446,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
28055,2025-04-25 13:43:39,2025-04-25 13:43:39,IR_60085.jpg,3,Cableduct,Cableduct,cableduct_2,22.009373,26.700001,-2.109251,...,0.500000,7.281390,3.346290,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
26953,2025-04-25 12:33:09,2025-04-25 12:33:09,IR_59944.jpg,6,Cableduct,Cableduct,cableduct_2,21.848385,29.200001,-2.091236,...,1.000000,5.510767,2.250796,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
24556,2025-04-25 10:17:39,2025-04-25 10:17:39,IR_59673.jpg,5,Cableduct,Cableduct,cableduct_2,23.445864,26.600000,2.079485,...,0.500000,4.810170,2.207242,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
28199,2025-04-25 13:52:38,2025-04-25 13:52:38,IR_60103.jpg,10,Cableduct,Cableduct,cableduct_2,24.455154,27.799999,2.067308,...,0.000000,6.356072,2.256669,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct


Hottest rows:


,timestamp_dt,timestamp,image_name,detection_index,label,canonical_label,object_id,temp_mean_c,temp_max_c,temp_delta_c,...,gap_min,projected_x,projected_y,anchor_x,anchor_y,display_x,display_y,position_source,static_cluster_id,clustering_method
28010,2025-04-25 13:41:09,2025-04-25 13:41:09,IR_60080.jpg,10,Cableduct,Cableduct,cableduct_2,24.697624,29.700001,1.425438,...,0.000000,5.227508,2.294672,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
27552,2025-04-25 13:10:09,2025-04-25 13:10:09,IR_60018.jpg,4,Cableduct,Cableduct,cableduct_2,24.579693,33.299999,1.801329,...,1.516667,6.891170,3.194609,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
27878,2025-04-25 13:32:38,2025-04-25 13:32:38,IR_60063.jpg,11,Cableduct,Cableduct,cableduct_2,24.575544,28.400000,1.643041,...,0.000000,6.336694,2.242172,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
27799,2025-04-25 13:27:38,2025-04-25 13:27:38,IR_60053.jpg,6,Cableduct,Cableduct,cableduct_2,24.558969,33.200001,2.443660,...,0.500000,7.338344,3.350806,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
28199,2025-04-25 13:52:38,2025-04-25 13:52:38,IR_60103.jpg,10,Cableduct,Cableduct,cableduct_2,24.455154,27.799999,2.067308,...,0.000000,6.356072,2.256669,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
28361,2025-04-25 14:01:39,2025-04-25 14:01:39,IR_60121.jpg,7,Cableduct,Cableduct,cableduct_2,24.196163,29.299999,0.833156,...,0.000000,7.481864,2.486051,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
28046,2025-04-25 13:43:09,2025-04-25 13:43:09,IR_60084.jpg,11,Cableduct,Cableduct,cableduct_2,24.118624,28.200001,1.088661,...,0.000000,7.287808,2.484919,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
27602,2025-04-25 13:13:39,2025-04-25 13:13:39,IR_60025.jpg,10,Cableduct,Cableduct,cableduct_2,24.016699,28.100000,1.252581,...,0.000000,5.085540,2.216485,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
26942,2025-04-25 12:32:09,2025-04-25 12:32:09,IR_59942.jpg,7,Cableduct,Cableduct,cableduct_2,23.939621,33.200001,1.503532,...,1.000000,7.199069,3.289493,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
28119,2025-04-25 13:47:39,2025-04-25 13:47:39,IR_60093.jpg,4,Cableduct,Cableduct,cableduct_2,23.815849,33.900002,0.684175,...,0.500000,7.194311,3.311062,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct


In [6]:
# Duplicate timestamp check: multiple rows for the same object and timestamp can create noisy GT curves later.
duplicate_groups = (
    object_day
    .groupby('timestamp_dt')
    .size()
    .reset_index(name='rows_at_timestamp')
    .query('rows_at_timestamp > 1')
    .sort_values('rows_at_timestamp', ascending=False)
)
print(f"Duplicate timestamps for {DEBUG_OBJECT_ID} on {DEBUG_DATE}: {len(duplicate_groups)}")
display(duplicate_groups.head(DEBUG_TOP_N))

if not duplicate_groups.empty:
    duplicated_rows = object_day[object_day['timestamp_dt'].isin(duplicate_groups['timestamp_dt'])].copy()
    duplicated_rows = duplicated_rows.sort_values(['timestamp_dt', 'detection_index'])
    duplicate_display_cols = [col for col in inspect_cols if col in duplicated_rows.columns]
    display(duplicated_rows[duplicate_display_cols].head(DEBUG_TOP_N * 3))


Duplicate timestamps for cableduct_2 on 2025-04-25: 63


,timestamp_dt,rows_at_timestamp
676,2025-04-25 13:49:09,3
682,2025-04-25 13:53:39,3
237,2025-04-25 09:35:39,3
2,2025-04-25 07:37:09,2
629,2025-04-25 13:07:39,2
652,2025-04-25 13:32:38,2
634,2025-04-25 13:13:39,2
630,2025-04-25 13:08:09,2
628,2025-04-25 13:06:39,2
662,2025-04-25 13:41:09,2


,timestamp_dt,timestamp,image_name,detection_index,label,canonical_label,object_id,temp_mean_c,temp_max_c,temp_delta_c,...,gap_min,projected_x,projected_y,anchor_x,anchor_y,display_x,display_y,position_source,static_cluster_id,clustering_method
21314,2025-04-25 07:37:09,2025-04-25 07:37:09,IR_59352.jpg,7,Cableduct,Cableduct,cableduct_2,19.719055,22.400000,-0.273522,...,0.500000,9.218672,3.074748,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
21315,2025-04-25 07:37:09,2025-04-25 07:37:09,IR_59352.jpg,11,Cableduct,Cableduct,cableduct_2,20.003307,22.400000,0.284252,...,0.000000,6.837590,3.156633,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
22847,2025-04-25 08:50:39,2025-04-25 08:50:39,IR_59499.jpg,8,Cableduct,Cableduct,cableduct_2,21.131323,24.000000,0.079388,...,0.500000,7.222908,3.324115,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
22848,2025-04-25 08:50:39,2025-04-25 08:50:39,IR_59499.jpg,11,Cableduct,Cableduct,cableduct_2,20.853014,24.000000,-0.278309,...,0.000000,5.622603,2.352094,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
22890,2025-04-25 08:52:39,2025-04-25 08:52:39,IR_59503.jpg,5,Cableduct,Cableduct,cableduct_2,21.398756,26.500000,0.148108,...,0.500000,7.232780,3.318498,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27529,2025-04-25 13:08:09,2025-04-25 13:08:09,IR_60014.jpg,6,Cableduct,Cableduct,cableduct_2,21.906839,29.299999,-1.601027,...,0.500000,5.693965,2.297709,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
27530,2025-04-25 13:08:09,2025-04-25 13:08:09,IR_60014.jpg,9,Cableduct,Cableduct,cableduct_2,22.562540,29.299999,0.655701,...,0.000000,7.251842,3.328636,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
27601,2025-04-25 13:13:39,2025-04-25 13:13:39,IR_60025.jpg,3,Cableduct,Cableduct,cableduct_2,22.764118,28.100000,0.072979,...,0.500000,6.768492,3.153079,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
27602,2025-04-25 13:13:39,2025-04-25 13:13:39,IR_60025.jpg,10,Cableduct,Cableduct,cableduct_2,24.016699,28.100000,1.252581,...,0.000000,5.085540,2.216485,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct


In [7]:
# Trace the selected object/date back to the 05 integrated CSV.
integrated_debug = pd.read_csv(config.INTEGRATED_OBJECTS_CSV)
integrated_debug['timestamp_dt'] = pd.to_datetime(integrated_debug['timestamp'])
integrated_debug = integrated_debug[
    (integrated_debug['object_id'] == DEBUG_OBJECT_ID) &
    (integrated_debug['timestamp_dt'].dt.date.astype(str) == DEBUG_DATE)
].copy()
for column in ['temp_mean_c', 'temp_max_c', 'projected_x', 'projected_y', 'anchor_x', 'anchor_y']:
    if column in integrated_debug.columns:
        integrated_debug[column] = pd.to_numeric(integrated_debug[column], errors='coerce')

print(f"Integrated rows for {DEBUG_DATE} | {DEBUG_OBJECT_ID}: {len(integrated_debug)}")
label_summary = (
    integrated_debug
    .groupby(['label', 'canonical_label', 'position_source', 'clustering_method'], dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values('rows', ascending=False)
)
display(label_summary)

integrated_cols = [
    'timestamp', 'image_name', 'detection_index', 'label', 'canonical_label', 'object_id',
    'temp_mean_c', 'temp_max_c', 'projected_x', 'projected_y', 'anchor_x', 'anchor_y',
    'display_x', 'display_y', 'position_source', 'static_cluster_id', 'clustering_method',
]
integrated_cols = [col for col in integrated_cols if col in integrated_debug.columns]
print('Integrated hottest rows:')
display(integrated_debug.sort_values('temp_mean_c', ascending=False)[integrated_cols].head(DEBUG_TOP_N))


Integrated rows for 2025-04-25 | cableduct_2: 964


,label,canonical_label,position_source,clustering_method,rows
0,Cableduct,Cableduct,dbscan_cableduct_global_median_anchor,dbscan_cableduct,862
1,Cableduct,Cableduct,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct,102


Integrated hottest rows:


,timestamp,image_name,detection_index,label,canonical_label,object_id,temp_mean_c,temp_max_c,projected_x,projected_y,anchor_x,anchor_y,display_x,display_y,position_source,static_cluster_id,clustering_method
28010,2025-04-25 13:41:09,IR_60080.jpg,10,Cableduct,Cableduct,cableduct_2,24.697624,29.700001,5.227508,2.294672,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
27552,2025-04-25 13:10:09,IR_60018.jpg,4,Cableduct,Cableduct,cableduct_2,24.579693,33.299999,6.891170,3.194609,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
27878,2025-04-25 13:32:38,IR_60063.jpg,11,Cableduct,Cableduct,cableduct_2,24.575544,28.400000,6.336694,2.242172,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
27799,2025-04-25 13:27:38,IR_60053.jpg,6,Cableduct,Cableduct,cableduct_2,24.558969,33.200001,7.338344,3.350806,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
28199,2025-04-25 13:52:38,IR_60103.jpg,10,Cableduct,Cableduct,cableduct_2,24.455154,27.799999,6.356072,2.256669,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
28361,2025-04-25 14:01:39,IR_60121.jpg,7,Cableduct,Cableduct,cableduct_2,24.196163,29.299999,7.481864,2.486051,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
28046,2025-04-25 13:43:09,IR_60084.jpg,11,Cableduct,Cableduct,cableduct_2,24.118624,28.200001,7.287808,2.484919,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
27602,2025-04-25 13:13:39,IR_60025.jpg,10,Cableduct,Cableduct,cableduct_2,24.016699,28.100000,5.085540,2.216485,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_nearest_same_label_anchor,dbscan_cableduct_0000,dbscan_cableduct
26942,2025-04-25 12:32:09,IR_59942.jpg,7,Cableduct,Cableduct,cableduct_2,23.939621,33.200001,7.199069,3.289493,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
28119,2025-04-25 13:47:39,IR_60093.jpg,4,Cableduct,Cableduct,cableduct_2,23.815849,33.900002,7.194311,3.311062,7.344941,3.358257,7.344941,3.358257,dbscan_cableduct_global_median_anchor,dbscan_cableduct_0000,dbscan_cableduct
